# MusicSage Separator — PyTorch (инференс)

Разделение музыки на стемы (drums / bass / other / vocals) с помощью
гибридной 2D+1D U-Net (HybridUNet).

Здесь только подключение обученной модели и её использование:
1. Загрузка треков MUSDB18 (test) через `musdb`;
2. Определение архитектуры HybridUNet;
3. Загрузка весов из чекпойнта `checkpoints_pytorch/sep_best.pt`;
4. Инференс: демо-разделение вокала, SI-SDR-метрика, разделение своего файла.


## Установка зависимостей

```bash
pip install torch musdb soundfile numpy
```

> `torchaudio` не нужен — STFT/ISTFT делаем через `torch.stft`/`torch.istft`.
> `musdb` тянет за собой `stempeg`, который читает `.stem.mp4` — нужен только
> `ffmpeg` в `$PATH`.

---


In [3]:
!pip install musdb #soundfile numpy torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.2/963.2 kB 61.8 MB/s eta 0:00:00


In [ ]:
import os

import musdb
import numpy as np
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)


In [ ]:
# ============================================================
# CONFIG
# ============================================================

SAMPLE_RATE = 32000          # чекпойнт обучен на 32 кГц
CHUNK_SEC = 5
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_SEC

# WaveUNet: stride-4 свёртки, LEVELS уровней => вход паддится до кратного 4**LEVELS
STRIDE = 4
LEVELS = 5
PAD_LEN = ((CHUNK_SAMPLES + STRIDE ** LEVELS - 1) // STRIDE ** LEVELS) * STRIDE ** LEVELS

EPS = 1e-4

# Корень MUSDB18: здесь лежат папки train/ и test/ с *.stem.mp4
DB_ROOT = "../root/MUSDB18/MUSDB18-7"

N_SOURCES = 4
SOURCE_NAMES = ["drums", "bass", "other", "vocals"]

# Спектральная ветка гибрида: STFT микса -> log-магнитуда -> маски
SPEC_NFFT = 4096
SPEC_HOP = 1024

# Параметры WaveUNet (должны совпадать с теми, на которых обучен чекпойнт)
WAVE_CHANNELS = 64
WAVE_LEVELS = 5

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("device:", DEVICE)
print("chunk:", CHUNK_SAMPLES, "samples | padded:", PAD_LEN)


In [15]:
def _stems_dict(track):
    """Все 5 стемов трека -> {mix, drums, bass, other, vocals} стерео."""
    stems = track.stems                       # (5, N, 2)
    return {
        "mix": stems[0].astype(np.float32),
        "drums": stems[1].astype(np.float32),
        "bass": stems[2].astype(np.float32),
        "other": stems[3].astype(np.float32),
        "vocals": stems[4].astype(np.float32),
    }


def load_song(track):
    """Полная песня: mix + 4 стема как стерео float32 (через API musdb)."""
    return _stems_dict(track)


In [ ]:
# Только тестовые треки — для демо и метрик ниже
mus_test = musdb.DB(root=DB_ROOT, subsets="test", sample_rate=SAMPLE_RATE)
print("test tracks:", len(mus_test))


## 3. Модель — HybridUNet

Гибрид 1D waveform U-Net (Demucs-стиль) + 2D U-Net по log-магнитуде STFT,
выход объединяется обучаемым гейтом. Инстанцирование и загрузка весов — ниже.


In [ ]:
_WINDOWS = {}


def get_window(n_fft, device):
    key = (n_fft, str(device))
    if key not in _WINDOWS:
        _WINDOWS[key] = torch.sqrt(
            torch.hann_window(n_fft, periodic=True, device=device)
        )
    return _WINDOWS[key]


def stft(x, fl, fs):
    """Комплексный STFT: (..., N) -> (..., F, T). Нужен только для loss."""
    shape = x.shape[:-1]
    s = torch.stft(
        x.reshape(-1, x.shape[-1]),
        n_fft=fl, hop_length=fs, win_length=fl,
        window=get_window(fl, x.device), return_complex=True,
    )
    return s.reshape(shape + s.shape[-2:])


class EncBlock(nn.Module):
    """Энкодер-блок: conv1d stride-4 + GLU (каналы x2 до GLU)."""
    def __init__(self, cin, cout, kernel=8, stride=4):
        super().__init__()
        self.conv = nn.Conv1d(cin, cout * 2, kernel, stride=stride,
                             padding=kernel // 2)
        self.glu = nn.GLU(dim=1)

    def forward(self, x):
        return self.glu(self.conv(x))


class Bottleneck(nn.Module):
    """Два dilated-conv слоя с GLU и residual-связью."""
    def __init__(self, channels, kernel=3):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels * 2, kernel, padding=1)
        self.glu1 = nn.GLU(dim=1)
        self.conv2 = nn.Conv1d(channels, channels * 2, kernel, padding=2, dilation=2)
        self.glu2 = nn.GLU(dim=1)

    def forward(self, x):
        return x + self.glu2(self.conv2(self.glu1(self.conv1(x))))


class DecBlock(nn.Module):
    """Декодер-блок: ConvTranspose stride-4 + GLU + skip + conv 3x3.

    Skip подрезается/допадывается до длины апсемпла, чтобы уровни
    совпадали по времени при любой длине входа.
    """
    def __init__(self, cin, cout, kernel=8, stride=4):
        super().__init__()
        self.deconv = nn.ConvTranspose1d(cin, cout * 2, kernel, stride=stride,
                                         padding=kernel // 2, output_padding=1)
        self.glu = nn.GLU(dim=1)
        self.conv = nn.Conv1d(cout, cout, 3, padding=1)

    def forward(self, x, skip=None):
        x = self.glu(self.deconv(x))
        if skip is not None:
            if x.shape[-1] > skip.shape[-1]:
                x = x[..., :skip.shape[-1]]
            elif x.shape[-1] < skip.shape[-1]:
                x = F.pad(x, (0, skip.shape[-1] - x.shape[-1]))
            x = x + skip
        return self.conv(x)


class WaveUNet(nn.Module):
    """1D waveform U-Net в стиле Demucs v2 (стерео).

    Вход: (B, 2, N) нормализованный микс; выход: (B, 4, 2, N) стемы
    (та же длина, что и вход). Стерео — это 2 канала входа, последний
    слой выдаёт n_sources * 2 каналов (пара на каждый стем).
    """

    def __init__(self, n_sources=N_SOURCES, channels=64, levels=5,
                 kernel=8, stride=4):
        super().__init__()
        self.stride = stride
        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()
        cin = 2
        for i in range(levels):
            cout = channels * (2 ** i)
            self.encoders.append(EncBlock(cin, cout, kernel, stride))
            cin = cout
        # декодеры: выход каналов = 2**max(0, i-1) * channels (под skip уровней),
        # верхний (i=levels-1) принимает bottleneck; нижний (i=0) — апсемпл без skip
        for i in range(levels):
            cout = channels * (2 ** max(0, i - 1))
            cin = channels * (2 ** i) if i == levels - 1 else channels * (2 ** max(0, i))
            self.decoders.append(DecBlock(cin, cout, kernel, stride))
        self.bottleneck = Bottleneck(channels * (2 ** (levels - 1)))
        self.head = nn.Conv1d(channels, n_sources * 2, 1)
        self.dummies = nn.Parameter(torch.zeros(n_sources))  # глобальный residual

    def forward(self, x):
        mix0 = x
        skips = []
        for enc in self.encoders:
            x = enc(x)
            skips.append(x)
        x = self.bottleneck(x)
        # скипы уровней: dec4 <- s3, ..., dec1 <- s0; нижний dec0 без skip
        for dec, skip in zip(self.decoders[::-1], skips[-2::-1]):
            x = dec(x, skip)
        x = self.decoders[0](x)
        if x.shape[-1] < mix0.shape[-1]:   # робастность к неделимым длинам
            x = F.pad(x, (0, mix0.shape[-1] - x.shape[-1]))
        x = self.head(x)[..., :mix0.shape[-1]]           # (B, S*2, N)
        x = x.view(x.shape[0], -1, 2, x.shape[-1])       # (B, S, 2, N)
        return x + (1 + self.dummies)[None, :, None, None] * mix0[:, None]


class SpecBlock(nn.Module):
    """2D conv-блок: conv3x3 -> GroupNorm -> SiLU -> conv3x3 -> GroupNorm + residual."""
    def __init__(self, cin, cout):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.gn1 = nn.GroupNorm(min(cout, 8), cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.gn2 = nn.GroupNorm(min(cout, 8), cout)
        self.act = nn.SiLU(inplace=True)
        self.shortcut = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x):
        h = self.act(self.gn1(self.conv1(x)))
        h = self.gn2(self.conv2(h))
        return self.act(h + self.shortcut(x))

class BiLSTMBottleneck(nn.Module):
    """BiLSTM по оси времени в bottleneck 2D U-Net.

    1x1 conv сжимает каналы в `hidden`, частота пулится до `f_pool` фиксированных
    бинов, признаки кадра складываются в один вектор, BiLSTM идёт по оси
    времени; затем частота восстанавливается и 1x1 conv возвращает каналы
    обратно. Residual-связь.
    """

    def __init__(self, channels, hidden=32, f_pool=16):
        super().__init__()
        self.hidden = hidden
        self.f_pool = f_pool
        self.proj_in = nn.Conv2d(channels, hidden, 1)
        self.lstm = nn.LSTM(hidden * f_pool, hidden * f_pool // 2, batch_first=True,
                           bidirectional=True)
        self.proj_out = nn.Conv2d(hidden, channels, 1)

    def forward(self, x):
        b, c, f, t = x.shape
        h = self.proj_in(x)                                  # (B, H, F, T)
        h = F.interpolate(h, size=(self.f_pool, t), mode="bilinear",
                          align_corners=False)               # (B, H, P, T)
        h = h.permute(0, 3, 1, 2).reshape(b, t, -1)          # (B, T, H*P)
        h, _ = self.lstm(h)                                  # (B, T, H)
        h = h.reshape(b, t, self.hidden, self.f_pool)
        h = h.permute(0, 2, 3, 1)                            # (B, H, P, T)
        h = F.interpolate(h, size=(f, t), mode="bilinear",
                          align_corners=False)               # (B, H, F, T)
        return self.proj_out(h) + x

class STFTBranch(nn.Module):
    """2D U-Net по log-магнитуде STFT микса -> 4 конкурентные маски.

    Вход: (B, 1, F, T); выход: (B, 4, F, T) маски в [0, 1], softmax по стемам
    на каждом T-F бине: сумма масок = 1, каждый бин делится между стемами
    «по-честному» (нет утечки other/vocals пополам).
    Частота и время даунсемплются в 2 раза на уровень.
    """
    def __init__(self, n_sources=N_SOURCES, base=16, levels=4):
        super().__init__()
        self.enc = nn.ModuleList()
        self.dec = nn.ModuleList()
        for i in range(levels):
            cin = 1 if i == 0 else base * 2 ** (i - 1)
            cout = base * 2 ** i
            self.enc.append(SpecBlock(cin, cout))
        c = base * 2 ** (levels - 1)
        self.bottleneck = BiLSTMBottleneck(c)
        for i in range(levels - 1, -1, -1):
            cout = base * 2 ** i
            cin = c + c if i == levels - 1 else base * 2 ** i * 3
            self.dec.append(SpecBlock(cin, cout))
        self.head = nn.Conv2d(base, n_sources, 1)

    def forward(self, spec):
        skips = []
        x = spec
        for enc in self.enc:
            x = enc(x)
            skips.append(x)
            x = F.avg_pool2d(x, 2)
        x = self.bottleneck(x)
        for dec, skip in zip(self.dec, skips[::-1]):
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear",
                              align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
        return self.head(x).softmax(dim=1)


class HybridUNet(nn.Module):
    """Гибрид: 1D waveform U-Net + 2D спектральные маски, объединение гейтом.

    Вход: (B, 2, N) нормализованный микс; выход: (B, 4, 2, N) стемы.

    - wave-ветка предсказывает стемы напрямую в waveform-домене;
    - spec-ветка строит мягкие маски по log-магнитуде STFT микса
      (канал микса обрабатывается независимо с общими весами), стемы
      восстанавливаются маскированием комплексного STFT + ISTFT;
    - выход = g * wave + (1 - g) * spec, где g — обучаемый per-stem гейт
      (init 0.5/0.5, модель сама учится, кому верить).
    """
    def __init__(self, n_sources=N_SOURCES, channels=64, levels=5,
                 nfft=4096, hop=1024, spec_base=16, spec_levels=4):
        super().__init__()
        self.wave = WaveUNet(n_sources=n_sources, channels=channels, levels=levels)
        self.spec = STFTBranch(n_sources=n_sources, base=spec_base, levels=spec_levels)
        self.nfft = nfft
        self.hop = hop
        self.gate = nn.Parameter(torch.zeros(n_sources))

    def forward(self, x):
        wav = self.wave(x)                             # (B, 4, 2, N)
        xf = x.float()
        win = torch.hann_window(self.nfft, device=x.device)
        st = torch.stft(xf.reshape(-1, x.shape[-1]), self.nfft, self.hop,
                        window=win, return_complex=True)   # (B*2, F, T)
        masks = self.spec(st.abs().log1p().unsqueeze(1))   # (B*2, 4, F, T)
        B = x.shape[0]
        spec_stems = torch.istft(
            (masks * st.unsqueeze(1)).view(B * 2 * N_SOURCES, *st.shape[-2:]),
            self.nfft, self.hop, window=win, length=x.shape[-1],
        )
        spec_stems = spec_stems.view(B, 2, N_SOURCES, x.shape[-1]) \
                                 .permute(0, 2, 1, 3)       # (B, 4, 2, N)
        g = torch.sigmoid(self.gate)[None, :, None, None]
        return (g * wav + (1 - g) * spec_stems).to(wav.dtype)




In [ ]:
# ============================================================
# Подключение модели: архитектура + веса чекпойнта
# ============================================================

CKPT_PATH = "checkpoints_pytorch/sep_best.pt"   # <-- путь к чекпойнту

model = HybridUNet(n_sources=N_SOURCES, channels=WAVE_CHANNELS,
                   levels=WAVE_LEVELS, nfft=SPEC_NFFT, hop=SPEC_HOP).to(DEVICE)

weights = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=True)
# Чекпойнт мог быть сохранён из nn.DataParallel-обёртки -> снимаем префикс "module."
if any(k.startswith("module.") for k in weights):
    weights = {k.removeprefix("module."): v for k, v in weights.items()}

model.load_state_dict(weights)
model.eval()
print(f"loaded {CKPT_PATH} | params: {sum(p.numel() for p in model.parameters()):,}")

# --- sanity check: прямой проход через сеть ---
with torch.no_grad():
    x0 = torch.randn(1, 2, PAD_LEN, device=DEVICE)
    out = model(x0)
print("input :", tuple(x0.shape))
print("output:", tuple(out.shape))

## 6. Инференс

Песню режем на перекрывающиеся чанки (50% overlap, кросфейд по краям),
предсказываем маску, реконструируем стем через ISTFT.

In [ ]:
def wiener_postprocess(stems, mix, gamma=2.0, fl=4096, fs=1024,
                        seg=30 * SAMPLE_RATE):
    """Wiener soft-mask пост-обработка (MUSDB 'wiki' трюк).

    Маски m_i = |S_i|^gamma / (sum_j |S_j|^gamma + EPS) считаются по STFT
    предсказанных стемов, итоговые стемы = ISTFT(m_i * STFT(mix)) с фазой
    микса. Сумма масок по стемам = 1, поэтому сумма стемов точно равна
    миксу (mixture consistency) и убирается шум/утечки между стемами.
    Считается сегментами по `seg` сэмплов, чтобы ограничить память.
    stems: (N, 4, 2), mix: (N, 2) -> (N, 4, 2) float32.
    """
    out = np.empty_like(stems)
    for start in range(0, len(mix), seg):
        e = stems[start:start + seg]
        m = mix[start:start + seg]
        s = stft(torch.from_numpy(np.ascontiguousarray(e.transpose((1, 2, 0)))).to(DEVICE),
                 fl, fs)                            # (4, 2, F, T)
        mst = stft(torch.from_numpy(np.ascontiguousarray(m.transpose((1, 0)))).to(DEVICE),
                   fl, fs)                          # (2, F, T)
        mag = s.abs().pow(gamma)                     # (4, 2, F, T)
        masks = mag / (mag.sum(dim=0, keepdim=True) + EPS)  # softmax по стемам
        est = (masks * mst.unsqueeze(0)).reshape(-1, *mst.shape[-2:])  # (4*2, F, T)
        r = torch.istft(est, fl, fs, window=get_window(fl, DEVICE), length=len(m))
        out[start:start + seg] = r.reshape(4, 2, len(m)).permute(2, 0, 1).cpu().numpy()
    return out


def separate_track_all(model, mix, device):
    """Полный трек -> все 4 стема (np.float32, (N, 4, 2)): перекрывающиеся
    чанки с Hann-кросфейдом + mixture-consistency проекция в каждом чанке
    и финальный Wiener soft-mask пост-процессинг по всему треку."""
    model.eval()
    n = len(mix)
    hop = CHUNK_SAMPLES // 2

    out = np.zeros((n, 4, 2), dtype=np.float32)
    weights = np.zeros((n, 2), dtype=np.float32)

    with torch.no_grad():
        for st in range(0, max(1, n - CHUNK_SAMPLES + 1), hop):
            chunk = mix[st:st + CHUNK_SAMPLES]            # (N', 2)
            if len(chunk) < CHUNK_SAMPLES:
                chunk = np.pad(chunk, ((0, CHUNK_SAMPLES - len(chunk)), (0, 0)))
            x = torch.from_numpy(chunk).to(device)        # (N, 2)
            scale = x.std(dim=0).clamp_min(1e-4)          # (2,)
            xp = F.pad((x / scale).t().unsqueeze(0),
                       (0, PAD_LEN - CHUNK_SAMPLES))      # (1, 2, N+pad)
            est = model(xp)[:, :, :, :CHUNK_SAMPLES] * scale[:, None]  # (1, 4, 2, N)
            est = est.squeeze(0).cpu().numpy().transpose(2, 0, 1)     # (N, 4, 2)

            # mixture consistency: остаток микса -> стемам по энергии
            residual = chunk - est.sum(1)                  # (N, 2)
            energy = (est ** 2).sum(axis=(0, 2), keepdims=True) + 1e-8  # (1, 4, 1)
            est = est + residual[:, None] * (energy / energy.sum())

            w = np.hanning(CHUNK_SAMPLES).astype(np.float32)  # Hann-кросфейд: 50% overlap => сумма весов соседних чанков = 1

            stop = min(st + CHUNK_SAMPLES, n)
            out[st:stop] += est[:stop - st] * w[:stop - st, None, None]
            weights[st:stop] += w[:stop - st, None]

    ok = weights[:, 0] > 0
    out[ok] /= weights[ok, None]
    return wiener_postprocess(out, mix)


def separate_track(model, mix, device, source_idx=0):
    """Полный трек -> стерео стем (np.float32, (N, 2)),
    перекрывающиеся чанки с кросфейдом."""
    model.eval()
    n = len(mix)
    hop = CHUNK_SAMPLES // 2

    out = np.zeros((n, 2), dtype=np.float32)
    weights = np.zeros(n, dtype=np.float32)

    with torch.no_grad():
        for st in range(0, max(1, n - CHUNK_SAMPLES + 1), hop):
            chunk = mix[st:st + CHUNK_SAMPLES]            # (N', 2)
            if len(chunk) < CHUNK_SAMPLES:
                chunk = np.pad(chunk, ((0, CHUNK_SAMPLES - len(chunk)), (0, 0)))
            x = torch.from_numpy(chunk).to(device)        # (N, 2)
            scale = x.std(dim=0).clamp_min(1e-4)          # (2,)
            xp = F.pad((x / scale).t().unsqueeze(0),
                       (0, PAD_LEN - CHUNK_SAMPLES))      # (1, 2, N+pad)
            est = model(xp)[0, source_idx, :, :CHUNK_SAMPLES] * scale[:, None]
            est = est.t().cpu().numpy()                   # (N, 2)

            w = np.hanning(CHUNK_SAMPLES).astype(np.float32)  # Hann-кросфейд: 50% overlap => сумма весов соседних чанков = 1

            stop = min(st + CHUNK_SAMPLES, n)
            out[st:stop] += est[:stop - st] * w[:stop - st, None]
            weights[st:stop] += w[:stop - st]

    ok = weights > 0
    out[ok] /= weights[ok, None]
    return out


def separate_file(model, track, device, source_idx=3):
    """Стем из musdb.Track (0 drums, 1 bass, 2 other, 3 vocals)."""
    song = load_song(track)
    stem = separate_track(model, song["mix"], device, source_idx=source_idx)
    return stem, song[SOURCE_NAMES[source_idx]], song["mix"]


In [ ]:
# Демо: отделить вокал из первого трека теста
test_track = mus_test[0]

est_vocals, ref_vocals, mix = separate_file(model, test_track, DEVICE, source_idx=3)

os.makedirs("output", exist_ok=True)
sf.write("output/mix.wav", mix, SAMPLE_RATE)
sf.write("output/vocals_ref.wav", ref_vocals, SAMPLE_RATE)
sf.write("output/vocals_pred.wav", est_vocals, SAMPLE_RATE)

print("wrote output/*.wav |", test_track.name)


## 7. Метрика — SI-SDR

Оценка качества отделённого стема на тестовых треках (SDR в дБ, чем больше — тем лучше).

In [ ]:
def si_sdr(estimate, reference):
    eps = 1e-8
    estimate = estimate - estimate.mean()
    reference = reference - reference.mean()
    alpha = np.dot(reference, estimate) / (np.dot(reference, reference) + eps)
    distortion = estimate - alpha * reference
    return 10 * np.log10(
        alpha ** 2 * np.dot(reference, reference)
        / (np.dot(distortion, distortion) + eps)
    )


def evaluate(model, tracks, max_songs=3, source_idx=3):
    sdr_list = []
    for track in tracks[:max_songs]:
        est, ref, _ = separate_file(model, track, DEVICE, source_idx=source_idx)
        n = min(len(est), len(ref))
        sdr_list.append(si_sdr(est[:n], ref[:n]))
        print(f"{track.name[:40]:42s} SI-SDR = {sdr_list[-1]:.1f} dB")
    return np.mean(sdr_list), sdr_list


mean_sdr, per_song = evaluate(model, mus_test, max_songs=3, source_idx=3)
print(f"\nmean SI-SDR (vocals): {mean_sdr:.1f} dB")


In [ ]:
# ============================================================
# Разделение своей песни
# ============================================================

import stempeg


def load_audio(path):
    """Любой аудиофайл (wav/mp3/flac/...) -> стерео float32 (N, 2), SAMPLE_RATE Гц.
    Моно-файлы дублируются в оба канала."""
    stems, _ = stempeg.read_stems(path, sample_rate=SAMPLE_RATE,
                                  ffmpeg_format="s16le")
    audio = np.squeeze(stems)               # (N,) или (N, C)
    if audio.ndim == 1:
        audio = np.stack([audio, audio], axis=1)
    elif audio.shape[1] != 2:
        audio = audio.mean(axis=1)
        audio = np.stack([audio, audio], axis=1)
    return audio.astype(np.float32)


# --- настройки ---
user_path = "my_song.mp3"                   # <-- путь к своему файлу
# Вместо in-memory модели можно загрузить чекпойнт:
# model.load_state_dict(torch.load("checkpoints_pytorch/sep_best.pt",
#                                  map_location=DEVICE))

if not os.path.exists(user_path):
    raise FileNotFoundError(f"файл не найден: {user_path}")

mix = load_audio(user_path)
print(f"{user_path}: {len(mix) / SAMPLE_RATE:.1f} s @ {SAMPLE_RATE} Hz")

def write_44k(path, data):
    """Запись wav в 44.1 кГц (ресемпл из SAMPLE_RATE через ffmpeg)."""
    import stempeg
    stempeg.write_audio(path, data, sample_rate=SAMPLE_RATE,
                        output_sample_rate=44100)


os.makedirs("output", exist_ok=True)
stems = separate_track_all(model, mix, DEVICE)   # (N, 4, 2), mixture consistency
for idx, name in enumerate(SOURCE_NAMES):
    write_44k(f"output/{name}.wav", stems[:, idx])
    print(f"saved output/{name}.wav")

write_44k("output/mix.wav", mix)
print("done")
